# labeled EarthCARE/MSI image chips


In [9]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import xarray as xr

SOURCE = Path("../step1_judge_for_msi/labeled_msi_data.nc")
raw = xr.open_dataset(SOURCE).load()

OUTPUT_ZARR = Path("output/earthcare-marine-stratocumulus-transitions/labeled_msi_data/earthcode_labeled_msi_data.zarr")
OUTPUT_ZARR.parent.mkdir(parents=True, exist_ok=True)

raw

<xarray.Dataset> Size: 280MB
Dimensions:     (sample: 533, channel: 4, y: 128, x: 128)
Coordinates:
  * sample      (sample) int64 4kB 0 5 3 2 3 5 7 5 6 9 ... 9 10 11 4 6 1 4 7 11
Dimensions without coordinates: channel, y, x
Data variables:
    unique_id   (sample) <U22 47kB '03401E_-13.6149_5.6921' ... '06426D_27.68...
    orbit       (sample) <U6 13kB '03401E' '03401E' ... '06426D' '06426D'
    region      (sample) <U3 6kB 'SEA' 'SEA' 'NEP' 'SEP' ... 'NEP' 'NEP' 'NEP'
    time_mid    (sample) datetime64[ns] 4kB 2025-01-02T13:30:15.431388992 ......
    lat_mid     (sample) float64 4kB -13.61 -20.91 24.44 ... 29.77 27.17 27.69
    lon_mid     (sample) float64 4kB 5.692 4.25 -117.5 ... -132.9 -133.5 -133.4
    ice_flag    (sample) float64 4kB 0.01764 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    label       (sample) float64 4kB 1.0 3.0 5.0 4.0 4.0 ... 1.0 2.0 2.0 2.0 2.0
    class_name  (sample) <U10 21kB 'Closed MCC' 'Disorganiz' ... 'Open MCC'
    image_data  (sample, channel, y, x) float64 279MB 0.4747 0.4003 ... 0.261

## Labels and coordinates


In [10]:
CLASS_NAMES = np.array(['Solid stratus', 'Closed MCC', 'Open MCC', 'Disorganized MCC', 'Clustered Cu', 'Suppressed Cu'])

In [11]:
ds = raw.drop_vars("class_name").assign_coords(
    sample=np.arange(raw.sizes["sample"], dtype="int32"),
    channel=['TIR3_minus_TIR2', 'TIR2_minus_TIR1', 'TIR2', 'COT'],
    y=np.arange(raw.sizes["y"], dtype="int16"),
    x=np.arange(raw.sizes["x"], dtype="int16"),
)
ds["source_sample_index"] = ("sample", raw.sample.values.astype("int16"))
ds["label"] = ("sample", raw.label.values.astype("int16"))
ds["label_name"] = ("sample", CLASS_NAMES[ds.label.values])
for name in ["unique_id", "orbit", "region"]:
    ds[name] = ("sample", raw[name].values.astype(str))
ds = ds.set_coords(["orbit", "region", "time_mid", "lat_mid", "lon_mid"])
ds


<xarray.Dataset> Size: 280MB
Dimensions:              (sample: 533, channel: 4, y: 128, x: 128)
Coordinates:
  * sample               (sample) int32 2kB 0 1 2 3 4 5 ... 528 529 530 531 532
  * channel              (channel) <U15 240B 'TIR3_minus_TIR2' ... 'COT'
  * y                    (y) int16 256B 0 1 2 3 4 5 ... 122 123 124 125 126 127
  * x                    (x) int16 256B 0 1 2 3 4 5 ... 122 123 124 125 126 127
    orbit                (sample) <U6 13kB '03401E' '03401E' ... '06426D'
    region               (sample) <U3 6kB 'SEA' 'SEA' 'NEP' ... 'NEP' 'NEP'
    time_mid             (sample) datetime64[ns] 4kB 2025-01-02T13:30:15.4313...
    lat_mid              (sample) float64 4kB -13.61 -20.91 ... 27.17 27.69
    lon_mid              (sample) float64 4kB 5.692 4.25 ... -133.5 -133.4
Data variables:
    unique_id            (sample) <U22 47kB '03401E_-13.6149_5.6921' ... '064...
    ice_flag             (sample) float64 4kB 0.01764 0.0 0.0 ... 0.0 0.0 0.0
    label                (sample) int16 1kB 1 3 5 4 4 4 3 3 ... 1 1 2 1 2 2 2 2
    image_data           (sample, channel, y, x) float64 279MB 0.4747 ... 0.261
    source_sample_index  (sample) int16 1kB 0 5 3 2 3 5 7 5 ... 11 4 6 1 4 7 11
    label_name           (sample) <U16 34kB 'Closed MCC' ... 'Open MCC'

## Metadata


In [12]:
ds.attrs['title'] = 'EarthCARE MSI labeled image chips for mesoscale cloud morphology classification'
ds.attrs['summary'] = 'Manual labels and normalized four-channel EarthCARE MSI-derived image chips for cloud morphology classification.'
ds.attrs['keywords'] = 'EarthCARE, MSI, stratocumulus, cloud morphology, closed cells, open cells, CNN, training data'
ds.attrs['license'] = 'CC-BY-4.0'
ds.attrs['platform'] = 'EarthCARE'
ds.attrs['instrument'] = 'Multi-Spectral Imager (MSI)'
ds.attrs['spatial_resolution'] = 'Approximately 1 km after 2x2 coarsening in preprocessing workflow; each chip is 128 x 128 local pixels.'
ds.attrs['processing_script'] = 'step1_judge_for_msi/fine_tune_judge_for_msi.ipynb'


In [13]:
# Added chip and label metadata.
ds['image_data'].attrs["long_name"] = 'normalized four-channel EarthCARE MSI image chip'
ds['image_data'].attrs["units"] = '1'
ds['image_data'].attrs["ancillary_variables"] = 'label label_name ice_flag'
ds['image_data'].attrs["comment"] = 'Channels are per-chip normalized TIR3-TIR2, TIR2-TIR1, TIR2, and cloud optical thickness. TIR channels come from EarthCARE MSI_NOM_1B pixel_values bands TIR1/TIR2/TIR3; cloud optical thickness comes from EarthCARE MSI_COP_2A cloud_optical_thickness.'
ds['channel'].attrs["long_name"] = 'model input channel'
ds['channel'].attrs["source_variables"] = 'MSI_NOM_1B pixel_values TIR1/TIR2/TIR3; MSI_COP_2A cloud_optical_thickness'
ds['channel'].attrs['description'] = "Derived MSI channels used by Johanna's CNN preprocessing: TIR3_minus_TIR2, TIR2_minus_TIR1, TIR2, and cloud optical thickness (per chip normalized)."
ds['label'].attrs["long_name"] = 'manual mesoscale cloud morphology class'
ds['label'].attrs["units"] = '1'
ds['label'].attrs["flag_values"] = np.arange(len(CLASS_NAMES), dtype='int16')
ds['label'].attrs["flag_meanings"] = ' '.join((name.lower().replace(' ', '_') for name in CLASS_NAMES))
ds['label_name'].attrs["long_name"] = 'human-readable class name derived from label'
ds['unique_id'].attrs["long_name"] = 'stable MSI chip identifier'
ds['unique_id'].attrs["comment"] = 'Format is orbit_latitude_longitude using the chip-centre latitude and longitude rounded by the original labeling workflow.'
ds['orbit'].attrs["long_name"] = 'EarthCARE orbit and frame identifier'
ds['orbit'].attrs["comment"] = 'Example format: 03401E.'
ds['region'].attrs["long_name"] = 'stratocumulus region code'
ds['region'].attrs["comment"] = str({'SEA': 'South East Atlantic stratocumulus region', 'NEA': 'North East Atlantic stratocumulus region', 'SEP': 'South East Pacific stratocumulus region', 'NEP': 'North East Pacific stratocumulus region'})
ds['time_mid'].attrs["long_name"] = 'chip centre acquisition time'
ds['time_mid'].attrs["standard_name"] = 'time'
ds['ice_flag'].attrs["long_name"] = 'fraction of chip pixels colder than 273 K in TIR2'
ds['ice_flag'].attrs["units"] = '1'
ds['ice_flag'].attrs["valid_min"] = 0.0
ds['ice_flag'].attrs["valid_max"] = 1.0
ds['source_sample_index'].attrs["long_name"] = 'original non-unique sample coordinate from source file'
ds['source_sample_index'].attrs["units"] = '1'
ds['sample'].attrs["long_name"] = 'unique image chip sample index'
ds['sample'].attrs["units"] = '1'

for axis, coordinate, units in [("lat", "latitude", "degrees_north"), ("lon", "longitude", "degrees_east")]:
    ds[f"{axis}_mid"].attrs["long_name"] = f"chip centre {coordinate}"
    ds[f"{axis}_mid"].attrs["standard_name"] = coordinate
    ds[f"{axis}_mid"].attrs["units"] = units
for axis, label in {"x": "column", "y": "row"}.items():
    ds[axis].attrs["long_name"] = f"chip-local image {label} pixel index"
    ds[axis].attrs["units"] = "1"
    ds[axis].attrs["comment"] = f"Local {axis} coordinate within the 128-pixel chip, not a projected geospatial coordinate."


## Save Zarr


In [14]:
ds.attrs["date_created"] = datetime.now(timezone.utc).isoformat()
for variable in ds.variables.values():
    variable.encoding["chunks"] = variable.shape

ds.to_zarr(OUTPUT_ZARR, mode="w", consolidated=True, zarr_format=2)

OUTPUT_ZARR


PosixPath('output/earthcare-marine-stratocumulus-transitions/labeled_msi_data/earthcode_labeled_msi_data.zarr')

## Check source values

In [ ]:
# Skip class_name (replaced by label_name) and the added sample/channel/x/y coordinates.
original = xr.open_dataset(SOURCE)
saved = xr.open_zarr(OUTPUT_ZARR, consolidated=True)
for name in original.drop_vars("class_name").variables:
    np.testing.assert_array_equal(
        saved["source_sample_index" if name == "sample" else name].values,
        original[name].values, err_msg=name,
    )
np.testing.assert_array_equal(saved.label_name.values, CLASS_NAMES[original.label.values.astype(int)])
